# 03. Análisis y visualización

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

In [12]:
df = pd.read_csv(
    "../data/processed/bank_clean.csv",
    parse_dates=["fecha_contacto", "fecha_alta"],
)
df.shape

(43000, 36)

In [13]:
# Columnas enteras con nulos: de float64 a Int64
columnas_int = ["edad", "dias_ultimo_contacto", "anio_contacto", "mes_contacto", "trimestre_contacto"]
for col in columnas_int:
    df[col] = df[col].astype("Int64")

# Categorías con orden lógico (para que las tablas y gráficos no salgan en orden alfabético)
orden_edad = ["<25", "25-34", "35-44", "45-54", "55-64", "65+", "desconocido"]
orden_ingresos = ["bajo", "medio-bajo", "medio-alto", "alto"]

df["grupo_edad"] = pd.Categorical(df["grupo_edad"], categories=orden_edad, ordered=True)
df["tramo_ingresos"] = pd.Categorical(df["tramo_ingresos"], categories=orden_ingresos, ordered=True)

In [14]:
def resumen_de_nulos(df):
    resumen = pd.DataFrame({
        "nulos": df.isnull().sum(),
        "porcentaje": (df.isnull().mean() * 100).round(1),
    })
    return resumen[resumen["nulos"] > 0].sort_values("nulos", ascending=False)

# resumen_de_nulos(df_bank)

In [15]:
df[columnas_int + ["grupo_edad", "tramo_ingresos"]].dtypes
resumen_de_nulos(df)

,nulos,porcentaje
dias_ultimo_contacto,41412,96.3
edad,5120,11.9
fecha_contacto,248,0.6
antiguedad_anios,248,0.6
anio_contacto,248,0.6
mes_contacto,248,0.6
trimestre_contacto,248,0.6


In [16]:
vars_numericas = [
    "edad", "duracion", "contactos_campana", "contactos_previos", "dias_ultimo_contacto",
    "ingresos", "visitas_web_mes", "total_hijos", "antiguedad_anios",
    "tasa_var_empleo", "indice_precios", "indice_confianza", "euribor_3m", "num_empleados",
]

vars_categoricas = [
    "trabajo", "estado_civil", "educacion", "hipoteca", "prestamo",
    "medio_contacto", "resultado_previo", "contactado_antes",
    "grupo_edad", "tramo_ingresos", "suscrito",
]

# Qué dejamos fuera y por qué:
# id_cliente: es un identificador, no se analiza.
# latitud y longitud: coordenadas de EE. UU., incoherentes con el enunciado. Ya está documentado.
# num_ninos y num_adolescentes: están recogidos en total_hijos.
# fecha_*, anio_*, mes_*, trimestre_*: se analizarán como serie temporal, no con estadísticos descriptivos.
# suscrito_num: es la versión numérica de suscrito.

In [17]:
descriptivo = df[vars_numericas].astype(float).describe().T
descriptivo["mediana"] = df[vars_numericas].astype(float).median()
descriptivo[["count", "mean", "mediana", "std", "min", "max"]].round(2)

,count,mean,mediana,std,min,max
edad,37880.0,39.98,38.00,10.44,17.00,98.00
duracion,43000.0,257.74,179.00,258.67,0.00,4918.00
contactos_campana,43000.0,2.57,2.00,2.77,1.00,56.00
contactos_previos,43000.0,0.17,0.00,0.50,0.00,7.00
dias_ultimo_contacto,1588.0,6.07,6.00,3.86,0.00,27.00
ingresos,43000.0,93241.20,93050.50,50498.32,5841.00,180802.00
visitas_web_mes,43000.0,16.59,17.00,9.24,1.00,32.00
total_hijos,43000.0,2.00,2.00,1.15,0.00,4.00
antiguedad_anios,42752.0,4.14,4.10,1.72,0.00,8.00
tasa_var_empleo,43000.0,0.08,1.10,1.57,-3.40,1.40


In [18]:
descriptivo["dif_media_mediana_%"] = ((descriptivo["mean"] - descriptivo["mediana"]) / descriptivo["mediana"] * 100).round(1)
descriptivo[["mean", "mediana", "dif_media_mediana_%"]].round(2)

,mean,mediana,dif_media_mediana_%
edad,39.98,38.00,5.2
duracion,257.74,179.00,44.0
contactos_campana,2.57,2.00,28.4
contactos_previos,0.17,0.00,inf
dias_ultimo_contacto,6.07,6.00,1.2
ingresos,93241.20,93050.50,0.2
visitas_web_mes,16.59,17.00,-2.4
total_hijos,2.00,2.00,0.2
antiguedad_anios,4.14,4.10,0.9
tasa_var_empleo,0.08,1.10,-93.0


In [19]:
descriptivo["coef_variacion_%"] = (descriptivo["std"] / descriptivo["mean"] * 100).round(1)
descriptivo[["std", "mean", "coef_variacion_%"]].round(2)

# No incluir estos resultados pues exceden el alcance del ejercicio.

,std,mean,coef_variacion_%
edad,10.44,39.98,26.1
duracion,258.67,257.74,100.4
contactos_campana,2.77,2.57,108.0
contactos_previos,0.50,0.17,285.8
dias_ultimo_contacto,3.86,6.07,63.6
ingresos,50498.32,93241.20,54.2
visitas_web_mes,9.24,16.59,55.7
total_hijos,1.15,2.00,57.6
antiguedad_anios,1.72,4.14,41.6
tasa_var_empleo,1.57,0.08,2040.6


### Interpretación de las variables numéricas
- `edad`: media X, mediana Y → distribución [simétrica / ligeramente asimétrica].
- `duracion`: media X frente a mediana Y → asimetría a la derecha (llamadas muy largas).
- `contactos_campana`: ...
- Variables económicas (`euribor_3m`, `num_empleados`...): ver nota.
- `ingresos`, `visitas_web_mes`, `total_hijos`: distribuciones muy uniformes (posible origen sintético).

Nota sobre las variables económicas: tasa_var_empleo, indice_precios, indice_confianza, euribor_3m y num_empleados no son atributos del cliente, sino del momento de la llamada. Muchas filas comparten el mismo valor (los “paquetes” de la Sesión 4), así que su media describe el periodo, no al cliente. Menciónalo en la interpretación.

In [20]:
def tabla_frecuencias(df, columna):
    frec = df[columna].value_counts(dropna=False)
    porc = (df[columna].value_counts(dropna=False, normalize=True) * 100).round(1)
    return pd.DataFrame({"frecuencia": frec, "porcentaje": porc})

In [23]:
for col in vars_categoricas:
    print(f"--- {col} ---")
    print(tabla_frecuencias(df, col))
    print()

--- trabajo ---
               frecuencia  porcentaje
trabajo                              
admin.              10873        25.3
blue-collar          9654        22.5
technician           7026        16.3
services             4162         9.7
management           3050         7.1
retired              1790         4.2
entrepreneur         1522         3.5
self-employed        1489         3.5
housemaid            1123         2.6
unemployed           1063         2.5
student               903         2.1
desconocido           345         0.8

--- estado_civil ---
              frecuencia  porcentaje
estado_civil                        
married            25999        60.5
single             12105        28.2
divorced            4811        11.2
desconocido           85         0.2

--- educacion ---
                     frecuencia  porcentaje
educacion                                  
university.degree         12722        29.6
high.school                9925        23.1
basic.9y     

### Interpretación de las variables categóricas
- Perfil dominante: ...
- Categorías minoritarias a tratar con cautela: ...
- Peso de "desconocido": ...
- Variable objetivo desbalanceada: 11,3 % de `yes`.

## Resumen de la Sesión 7
- Dataset cargado desde `bank_clean.csv` con `parse_dates` y tipos restaurados.
- Estadísticos de 14 variables numéricas (media, mediana, desviación, mín., máx.).
- Frecuencias de 11 variables categóricas con la función `tabla_frecuencias`.
- Interpretaciones escritas.

# Interpretación de variables 

### Interpretación de las variables numéricas

**Variables del cliente y de la llamada**
- `edad`: media 40,0 y mediana 38 años (diferencia del 5,2 %). Distribución ligeramente asimétrica hacia edades altas, con máximo de 98. El 11,9 % de las filas no tiene edad. Cliente típico: unos 38 años.
- `duracion`: media 257,7 s frente a mediana 179 s (diferencia del 44 %). Fuerte asimetría a la derecha: la mayoría de las llamadas son cortas y unas pocas muy largas (máximo 4.918 s) elevan la media. La **mediana representa mejor la llamada típica** (unos 3 minutos). Se conservaron los outliers, por lo que se usará la mediana junto a la media.
- `contactos_campana`: media 2,57 y mediana 2 (diferencia del 28 %). Cola larga hasta 56 contactos. El cliente típico recibe 2 contactos; unos pocos reciben muchos más.
- `contactos_previos`: media 0,17 y mediana 0. La gran mayoría (86,3 %) no tuvo campañas previas, por lo que la variable es casi siempre 0. La diferencia relativa no es calculable (mediana 0).
- `dias_ultimo_contacto`: media 6,1 y mediana 6 días. Es simétrica y compacta, pero solo tiene dato en 1.588 clientes (3,7 %), casi todos con éxito previo. **No es representativa** de todos los contactados antes.

**Variables demográficas (Excel)**
- `ingresos` (media 93.241 y mediana 93.051), `visitas_web_mes` (16,6 y 17), `total_hijos` (2,0 y 2) y `antiguedad_anios` (4,1 y 4,1): diferencias inferiores al 3 %, es decir, distribuciones muy simétricas. Su carácter tan uniforme apunta a un **posible origen sintético**, por lo que no se sacarán conclusiones fuertes de ellas.

**Variables del contexto económico**
- `tasa_var_empleo`, `indice_precios`, `indice_confianza`, `euribor_3m` y `num_empleados` no son atributos del cliente, sino del **momento de la llamada**: muchos registros comparten el mismo valor. Sus medias describen el periodo, no al cliente.
- `euribor_3m`: media 3,62 y mediana 4,86 (diferencia del -25,6 %). Distribución con dos bloques, uno de valores altos (≈ 4,8-5,0) y otro de valores bajos (desde 0,63), lo que refleja dos etapas económicas distintas. Recordar que el 21,5 % de sus valores fue imputado.
- `tasa_var_empleo`: media 0,08 y mediana 1,10. También refleja periodos de contextos opuestos (de -3,4 a +1,4).
- `indice_precios`, `indice_confianza` y `num_empleados`: variación baja respecto a su media (desviación pequeña), lo que indica un contexto relativamente estable en estas tres.

### Interpretación de las variables categóricas

**Perfil dominante**
- `trabajo`: `admin.` (25,3 %), `blue-collar` (22,5 %) y `technician` (16,3 %) suman el 64 % de los clientes. Los perfiles minoritarios son `student` (2,1 %) y `unemployed` (2,5 %).
- `estado_civil`: mayoría de casados (60,5 %), seguidos de solteros (28,2 %) y divorciados (11,2 %).
- `educacion`: predominan los estudios universitarios (29,6 %) y secundarios (23,1 %); entre ambos superan el 52 %.
- `hipoteca`: reparto casi mitad y mitad (52,3 % sí, 45,3 % no). `prestamo`: solo el 15,2 % tiene otro préstamo.
- `medio_contacto`: el 63,7 % de los contactos se hizo por móvil (`cellular`) y el 36,3 % por teléfono fijo.
- `grupo_edad`: concentración en 25-34 años (29,5 %) y 35-44 años (28,8 %); entre ambos y los 45-54 suman casi el 77 % de la muestra. Son muy minoritarios los menores de 25 (2,3 %) y los mayores de 65 (1,4 %).
- `tramo_ingresos`: 25 % en cada tramo, por construcción (cuartiles).

**Historial previo**
- `resultado_previo`: el 86,3 % no tuvo campaña previa (`nonexistent`), el 10,4 % fracasó y solo el 3,3 % acabó en éxito. `contactado_antes` refleja lo mismo: 13,7 % contactados antes.

**Categorías minoritarias a tratar con cautela**
- `educacion = illiterate`: solo 18 casos (0,04 %). Cualquier tasa de conversión sobre ellos será inestable y no se interpretará, o se agrupará con `basic.4y`.
- `grupo_edad` `<25` (994) y `65+` (616): grupos pequeños, comparar con prudencia.
- `resultado_previo = success`: 1.436 casos; suficientes, pero hay que recordar que sobre ellos se basa casi toda la información de `dias_ultimo_contacto`.

**Peso de "desconocido"**
- Muy bajo en `trabajo` (0,8 %), `estado_civil` (0,2 %) e `hipoteca`/`prestamo` (2,4 %); moderado en `educacion` (4,2 %); y **relevante en `grupo_edad` (11,9 %)**, porque procede de los nulos de `edad`.
  Con estos porcentajes, "desconocido" no distorsiona el análisis, salvo en edad, donde se comentarán los resultados con cautela.

**Variable objetivo**
- `suscrito`: 11,3 % de `yes` (4.844) frente a 88,7 % de `no` (38.156). Está **desbalanceada**: una tasa del 11,3 % es la referencia con la que compararemos cada segmento. Un grupo con una tasa del 20 % sería casi el doble de la media.

Los nulos en edad, fecha_contacto y dias_ultimo_contacto se conservaron deliberadamente: no existe un método fiable para deducirlos y imputarlos habría introducido datos artificiales. Cada análisis indica sobre cuántos registros se calcula.